# Análisis exploratorio del dataset de demanda eléctrica del SIN

Objetivo: conocer y caracterizar el dataset que se utilizará para el pronóstico de demanda eléctrica del Sistema Interconectado Nacional (SIN).

En este notebook se realiza:

- carga y copia de trabajo del dataset original;
- revisión de estructura, tipos de datos y valores nulos;
- identificación y eliminación de duplicados exactos;
- conversión de fechas a UTC y horario local de Paraguay;
- revisión de cambios horarios y discontinuidades temporales;
- estadísticas descriptivas, percentiles e IQR;
- visualizaciones iniciales de la demanda y sus patrones temporales.


## 1. Importación de librerías


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)


## 2. Carga del dataset

La celda permite trabajar tanto en Google Colab como en un entorno local. Si el archivo `complete-dataset.csv` no está disponible en la carpeta de trabajo, se solicita subirlo manualmente.


In [ ]:
DATA_FILE = Path("complete-dataset.csv")

if not DATA_FILE.exists():
    try:
        from google.colab import files

        uploaded = files.upload()
        if uploaded:
            DATA_FILE = Path(next(iter(uploaded.keys())))
    except ModuleNotFoundError:
        pass

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "No se encontró el archivo CSV. Verificá que 'complete-dataset.csv' esté en la carpeta "
        "del notebook o subilo cuando Colab lo solicite."
    )

df_raw = pd.read_csv(DATA_FILE)
df = df_raw.copy()

print(f"Archivo utilizado: {DATA_FILE}")
print(f"Dimensiones originales: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")


## 3. Inspección general


In [ ]:
print("Primeras filas:")
display(df.head())

print("Últimas filas:")
display(df.tail())

resumen_estructura = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "nulos_%": (df.isna().mean().values * 100).round(3),
    "valores_unicos": df.nunique(dropna=False).values,
})

display(resumen_estructura)


### Diccionario rápido de variables

Esta tabla ayuda a dejar documentado qué representa cada columna antes de continuar con la limpieza y el análisis.


In [ ]:
diccionario_variables = pd.DataFrame(
    {
        "variable": ["DATETIME", "SIN", "T02M", "RH2M", "PRSS", "TPP6", "U10M", "V10M", "ISHOLIDAY"],
        "descripcion": [
            "Fecha y hora del registro con offset horario",
            "Demanda eléctrica del SIN",
            "Temperatura a 2 metros",
            "Humedad relativa a 2 metros",
            "Presión atmosférica",
            "Precipitación acumulada en 6 horas",
            "Componente U del viento a 10 metros",
            "Componente V del viento a 10 metros",
            "Indicador de feriado",
        ],
    }
)

display(diccionario_variables)


## 4. Revisión temporal y conversión de fechas


In [ ]:
# Se conserva el texto original de DATETIME y se agregan dos representaciones temporales:
# una absoluta en UTC y otra en horario local de Paraguay.
df["DATETIME"] = df["DATETIME"].astype(str)
df["DATETIME_UTC"] = pd.to_datetime(df["DATETIME"], utc=True, errors="coerce")
df["DATETIME_LOCAL"] = df["DATETIME_UTC"].dt.tz_convert("America/Asuncion")
df["OFFSET"] = df["DATETIME"].str[-6:]

print("Primer DATETIME original:", df["DATETIME"].iloc[0])
print("Último DATETIME original:", df["DATETIME"].iloc[-1])
print("Fechas no interpretadas:", df["DATETIME_UTC"].isna().sum())

print("\nOffsets presentes:")
display(df["OFFSET"].value_counts().rename_axis("offset").reset_index(name="registros"))

display(df[["DATETIME", "DATETIME_UTC", "DATETIME_LOCAL", "OFFSET"]].head())


## 5. Duplicados y limpieza básica


In [ ]:
duplicados_exactos = df.duplicated().sum()
duplicados_datetime = df["DATETIME"].duplicated().sum()
duplicados_utc = df["DATETIME_UTC"].duplicated().sum()

print(f"Filas completamente duplicadas: {duplicados_exactos:,}")
print(f"DATETIME repetidos: {duplicados_datetime:,}")
print(f"DATETIME_UTC repetidos: {duplicados_utc:,}")

if duplicados_exactos > 0:
    print("\nEjemplos de filas duplicadas:")
    display(df[df.duplicated(keep=False)].head(10))


In [ ]:
# Se eliminan únicamente duplicados exactos para no modificar registros que puedan
# tener la misma fecha pero valores distintos.
df = df.drop_duplicates().reset_index(drop=True)

print(f"Dimensiones después de eliminar duplicados exactos: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")
print("Duplicados exactos restantes:", df.duplicated().sum())
print("DATETIME_UTC repetidos restantes:", df["DATETIME_UTC"].duplicated().sum())


## 6. Orden temporal, nulos y frecuencia de muestreo


In [ ]:
df = df.sort_values("DATETIME_UTC").reset_index(drop=True)

print("Registros originales:", len(df_raw))
print("Registros de trabajo:", len(df))

print("\nValores nulos por columna:")
display(df.isna().sum().rename("nulos").to_frame())

print("\nPrimer instante local:", df["DATETIME_LOCAL"].iloc[0])
print("Último instante local:", df["DATETIME_LOCAL"].iloc[-1])


In [ ]:
df["DELTA_UTC"] = df["DATETIME_UTC"].diff()

resumen_deltas = (
    df["DELTA_UTC"]
    .value_counts()
    .sort_index()
    .rename_axis("diferencia_entre_registros")
    .reset_index(name="cantidad")
)

display(resumen_deltas)


In [ ]:
discontinuidades = df[
    df["DELTA_UTC"].notna()
    & (df["DELTA_UTC"] != pd.Timedelta(hours=1))
].copy()

discontinuidades["horas_faltantes_estimadas"] = (
    discontinuidades["DELTA_UTC"] / pd.Timedelta(hours=1) - 1
).astype(int)

print("Cantidad de discontinuidades UTC:", len(discontinuidades))
print("Horas faltantes estimadas:", discontinuidades["horas_faltantes_estimadas"].sum())

display(
    discontinuidades[
        ["DATETIME", "DATETIME_UTC", "DATETIME_LOCAL", "DELTA_UTC", "horas_faltantes_estimadas"]
    ]
)


In [ ]:
# Verificación adicional: se construye el rango horario esperado en UTC y se comparan
# sus marcas temporales con las disponibles en el dataset.
rango_utc_esperado = pd.date_range(
    start=df["DATETIME_UTC"].min(),
    end=df["DATETIME_UTC"].max(),
    freq="h",
    tz="UTC",
)

faltantes_utc = rango_utc_esperado.difference(pd.DatetimeIndex(df["DATETIME_UTC"]))

print("Registros esperados en una serie horaria continua:", len(rango_utc_esperado))
print("Registros disponibles después de limpieza:", len(df))
print("Marcas UTC faltantes:", len(faltantes_utc))

display(pd.DataFrame({"DATETIME_UTC_faltante": faltantes_utc}).head(20))


## 7. Cambios de offset horario


In [ ]:
cambios_offset = df[df["OFFSET"] != df["OFFSET"].shift(1)].copy()

display(cambios_offset[["DATETIME", "DATETIME_UTC", "DATETIME_LOCAL", "OFFSET", "SIN"]])


### Interpretación preliminar

Las discontinuidades UTC y los cambios de offset deben revisarse juntos. Si las discontinuidades coinciden con cambios de horario oficial, no necesariamente indican errores de medición; aun así, deben documentarse porque afectan la construcción de rezagos, ventanas móviles y validaciones temporales.


## 8. Estadísticas descriptivas


In [ ]:
columnas_numericas = df.select_dtypes(include="number").columns.tolist()

estadisticas = df[columnas_numericas].describe().T
estadisticas["rango"] = estadisticas["max"] - estadisticas["min"]
estadisticas["coef_variacion"] = estadisticas["std"] / estadisticas["mean"].replace(0, np.nan)

display(estadisticas)


In [ ]:
percentiles = df[columnas_numericas].quantile(
    [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

percentiles.columns = [f"p{int(p * 100):02d}" for p in percentiles.columns]
display(percentiles)


## 9. IQR y posibles valores atípicos


In [ ]:
q1 = df[columnas_numericas].quantile(0.25)
q3 = df[columnas_numericas].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers_iqr = (
    (df[columnas_numericas] < limite_inferior)
    | (df[columnas_numericas] > limite_superior)
)

resumen_iqr = pd.DataFrame({
    "q1": q1,
    "q3": q3,
    "iqr": iqr,
    "limite_inferior": limite_inferior,
    "limite_superior": limite_superior,
    "posibles_outliers": outliers_iqr.sum(),
    "posibles_outliers_%": (outliers_iqr.mean() * 100).round(3),
})

display(resumen_iqr.sort_values("posibles_outliers", ascending=False))


> Nota: en series temporales de demanda eléctrica, un valor extremo no debe eliminarse automáticamente. Primero se debe revisar si corresponde a un evento real, un feriado, una ola de calor/frío, un cambio horario o un problema de registro.


## 10. Variables temporales para análisis gráfico


In [ ]:
df["ANIO"] = df["DATETIME_LOCAL"].dt.year
df["MES"] = df["DATETIME_LOCAL"].dt.month
df["DIA_SEMANA"] = df["DATETIME_LOCAL"].dt.dayofweek
df["HORA"] = df["DATETIME_LOCAL"].dt.hour
df["FECHA"] = df["DATETIME_LOCAL"].dt.date

display(df[["DATETIME_LOCAL", "ANIO", "MES", "DIA_SEMANA", "HORA", "FECHA"]].head())


## 11. Gráficos de demanda


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df["DATETIME_LOCAL"], df["SIN"], linewidth=0.7)
ax.set_title("Demanda eléctrica del SIN")
ax.set_xlabel("Fecha")
ax.set_ylabel("Demanda")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["SIN"], bins=60, edgecolor="white")
axes[0].set_title("Distribución de la demanda")
axes[0].set_xlabel("SIN")
axes[0].set_ylabel("Frecuencia")

axes[1].boxplot(df["SIN"].dropna(), vert=False)
axes[1].set_title("Boxplot de la demanda")
axes[1].set_xlabel("SIN")

plt.tight_layout()
plt.show()


In [ ]:
perfil_horario = df.groupby("HORA")["SIN"].agg(["mean", "median", "min", "max"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(perfil_horario.index, perfil_horario["mean"], marker="o", label="Media")
ax.plot(perfil_horario.index, perfil_horario["median"], marker="o", label="Mediana")
ax.set_title("Perfil horario promedio de la demanda")
ax.set_xlabel("Hora local")
ax.set_ylabel("SIN")
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.show()

display(perfil_horario)


In [ ]:
demanda_diaria = df.groupby("FECHA")["SIN"].agg(["mean", "max", "min"]).reset_index()
demanda_diaria["FECHA"] = pd.to_datetime(demanda_diaria["FECHA"])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(demanda_diaria["FECHA"], demanda_diaria["mean"], linewidth=0.8, label="Media diaria")
ax.plot(demanda_diaria["FECHA"], demanda_diaria["max"], linewidth=0.6, alpha=0.7, label="Máxima diaria")
ax.set_title("Demanda diaria media y máxima")
ax.set_xlabel("Fecha")
ax.set_ylabel("SIN")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
promedio_mensual = df.groupby(["ANIO", "MES"])["SIN"].mean().reset_index()
promedio_mensual["PERIODO"] = pd.to_datetime(
    promedio_mensual["ANIO"].astype(str) + "-" + promedio_mensual["MES"].astype(str) + "-01"
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(promedio_mensual["PERIODO"], promedio_mensual["SIN"], marker="o", markersize=3)
ax.set_title("Promedio mensual de demanda")
ax.set_xlabel("Mes")
ax.set_ylabel("SIN promedio")
plt.tight_layout()
plt.show()


## 12. Relación entre demanda y variables meteorológicas


In [ ]:
variables_correlacion = ["SIN", "T02M", "RH2M", "PRSS", "TPP6", "U10M", "V10M", "ISHOLIDAY"]
corr = df[variables_correlacion].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Matriz de correlación")

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(corr)


## 13. Cierre del EDA

Puntos que deben quedar documentados después de ejecutar el notebook:

- cantidad de registros originales y registros después de limpieza;
- rango temporal disponible;
- cantidad de duplicados eliminados;
- cantidad de marcas temporales faltantes o discontinuidades;
- comportamiento general de la demanda;
- variables con posibles valores atípicos;
- patrones horarios, diarios o mensuales relevantes;
- relación inicial entre demanda, clima y feriados.
